# Google Colab Backtesting Walkthrough

This notebook guides you through packaging, uploading, and backtesting your OpenAlgo strategies in Google Colab. Follow each section for a complete workflow.

## 1. Generate and Export Strategy Package

Run the following command in your terminal to package all strategies and utilities for Colab:

```bash
python strategies/scripts/export_to_colab.py
```

This will create `openalgo_colab.zip` in your project root, containing everything needed for Colab backtesting.

## 2. Upload Package to Google Colab

**Option A: Browser Upload**
- Open Google Colab and create a new notebook or use the one provided in the zip.
- Open the Files sidebar (folder icon).
- Upload `openalgo_colab.zip`.

**Option B: VS Code Extension**
- Install the Google Colab extension in VS Code.
- Open this notebook locally in VS Code.
- Connect to a Colab runtime.
- Run the following code to upload the zip file if the file explorer is not available:

In [ ]:
from google.colab import files
files.upload()  # Select openalgo_colab.zip from the popup

## 3. Extract and Prepare Environment

Unzip the package and install required dependencies for your strategies and utilities.

In [ ]:
import zipfile
import os

# Unzip the package
if os.path.exists('openalgo_colab.zip'):
    with zipfile.ZipFile('openalgo_colab.zip', 'r') as zip_ref:
        zip_ref.extractall('.')
else:
    print('openalgo_colab.zip not found. Please upload the package.')

# Install requirements if present
if os.path.exists('requirements.txt'):
    !pip install -r requirements.txt
elif os.path.exists('openalgo_colab/requirements.txt'):
    !pip install -r openalgo_colab/requirements.txt

## 4. Auto-Upload Handling in Notebook

Automatically prompt for upload if `openalgo_colab.zip` is missing.

In [ ]:
import os
from google.colab import files

if not os.path.exists('openalgo_colab.zip'):
    print('openalgo_colab.zip not found. Please upload the package.')
    files.upload()

## 5. Load Strategies and Utilities

Dynamically import all strategies from `strategies/scripts/*.py` and utilities from `strategies/utils`.

In [ ]:
import glob
import importlib.util
import sys

# Add strategies and utils to path
sys.path.append('strategies/scripts')
sys.path.append('strategies/utils')

# Dynamically import all strategy scripts
strategy_files = glob.glob('strategies/scripts/*.py')
strategies = {}
for file in strategy_files:
    name = os.path.splitext(os.path.basename(file))[0]
    spec = importlib.util.spec_from_file_location(name, file)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    strategies[name] = module

## 6. Data Fallback with yfinance

Fetch data from Yahoo Finance using `yfinance` if local data is unavailable.

In [ ]:
import yfinance as yf
import pandas as pd

def get_data(symbol, start, end):
    local_path = f'data/{symbol}.csv'
    if os.path.exists(local_path):
        return pd.read_csv(local_path)
    else:
        print(f'Fetching {symbol} data from Yahoo Finance...')
        df = yf.download(symbol, start=start, end=end)
        df.to_csv(local_path)
        return df

## 7. Interactive Backtest UI

Use ipywidgets to select strategy, symbol, and date range for backtesting.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

strategy_dropdown = widgets.Dropdown(
    options=list(strategies.keys()),
    description='Strategy:',
)
symbol_text = widgets.Text(
    value='RELIANCE',
    description='Symbol:',
)
date_range = widgets.DatePicker(
    description='Start Date',
)
end_date = widgets.DatePicker(
    description='End Date',
)
run_button = widgets.Button(description='Run Backtest')

ui = widgets.VBox([strategy_dropdown, symbol_text, date_range, end_date, run_button])
display(ui)

## 8. Run Backtest and Visualize Results

Execute the selected strategy and visualize equity curve, drawdown, and trade markers on candlestick charts.

In [ ]:
import matplotlib.pyplot as plt
import plotly.graph_objs as go
from IPython.display import clear_output

# Example backtest runner (replace with your strategy's run logic)
def run_backtest(strategy_name, symbol, start, end):
    df = get_data(symbol, start, end)
    strategy = strategies[strategy_name]
    # Assume strategy has a run_backtest(df) method
    results = strategy.run_backtest(df)
    return results

def on_run_clicked(b):
    clear_output(wait=True)
    display(ui)
    strategy_name = strategy_dropdown.value
    symbol = symbol_text.value
    start = date_range.value
    end = end_date.value
    results = run_backtest(strategy_name, symbol, start, end)
    # Plot equity curve
    plt.figure(figsize=(10,4))
    plt.plot(results['equity_curve'])
    plt.title('Equity Curve')
    plt.show()
    # Plot drawdown
    plt.figure(figsize=(10,4))
    plt.plot(results['drawdown'])
    plt.title('Drawdown')
    plt.show()
    # Candlestick with trade markers (using plotly)
    fig = go.Figure(data=[go.Candlestick(x=results['df'].index,
                                         open=results['df']['Open'],
                                         high=results['df']['High'],
                                         low=results['df']['Low'],
                                         close=results['df']['Close'])])
    for trade in results['trades']:
        fig.add_trace(go.Scatter(x=[trade['date']], y=[trade['price']],
                                 mode='markers', marker=dict(color='red' if trade['type']=='sell' else 'green', size=10),
                                 name=trade['type']))
    fig.show()

run_button.on_click(on_run_clicked)

## 9. Verification of Package and Notebook

Verify that `openalgo_colab.zip` contains the correct directory structure and that this notebook is present and correctly structured.

In [ ]:
import zipfile

with zipfile.ZipFile('openalgo_colab.zip', 'r') as zip_ref:
    print('Contents of openalgo_colab.zip:')
    zip_ref.printdir()
    assert 'strategies/scripts/' in [f.filename for f in zip_ref.filelist], 'strategies/scripts missing!'
    assert 'strategies/utils/' in [f.filename for f in zip_ref.filelist], 'strategies/utils missing!'
    assert 'Colab_Backtest.ipynb' in [f.filename for f in zip_ref.filelist], 'Colab_Backtest.ipynb missing!'
print('Verification complete.')

## Automated Backtest: All Strategies

This cell will run backtests for all detected strategies and summarize the results.

In [ ]:
import datetime
summary_results = []

# Set default symbol and date range (customize as needed)
default_symbol = 'RELIANCE'
default_start = datetime.date(2025, 1, 1)
default_end = datetime.date(2026, 2, 1)

for strategy_name, strategy_module in strategies.items():
    print(f'Running backtest for: {strategy_name}')
    try:
        df = get_data(default_symbol, default_start, default_end)
        # Assume each strategy has a run_backtest(df) method
        results = strategy_module.run_backtest(df)
        summary = {
            'strategy': strategy_name,
            'final_equity': results['equity_curve'][-1] if 'equity_curve' in results else None,
            'max_drawdown': min(results['drawdown']) if 'drawdown' in results else None,
            'total_trades': len(results['trades']) if 'trades' in results else None
        }
        summary_results.append(summary)
    except Exception as e:
        print(f'Error running {strategy_name}: {e}')
        summary_results.append({'strategy': strategy_name, 'error': str(e)})

import pandas as pd
summary_df = pd.DataFrame(summary_results)
display(summary_df)